# Oil News Project Demonstration
This notebook compiles the project scripts into a single run-through for easy demonstration.


## 1. Database Configuration
First, we load the database configuration.


In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class MySQLConfig:
    host: str
    port: int
    user: str
    password: str
    database: str


def load_dotenv(path: Path = Path(".env")) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def get_mysql_config() -> MySQLConfig:
    load_dotenv()
    return MySQLConfig(
        host=os.getenv("MYSQL_HOST", "127.0.0.1"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD", ""),
        database=os.getenv("MYSQL_DATABASE", "oil_news_project"),
    )



## 2. Load Datasets into MySQL
Load CSV datasets into the MySQL database using the config.


In [ ]:
from __future__ import annotations

import argparse
import csv
import re
from collections import defaultdict
from pathlib import Path
from typing import Iterable

from db_config import get_mysql_config


DATASET_DIR = Path("datasets")

DATE_COLUMNS = {
    "trade_date",
    "event_date",
    "gpr_date",
    "month_start",
    "snapshot_date",
    "market_date",
    "full_date",
}


def require_connector():
    try:
        import mysql.connector  # type: ignore
    except ModuleNotFoundError as exc:
        raise SystemExit(
            "mysql-connector-python is required for loading MySQL. "
            "Install with: python -m pip install -r requirements.txt"
        ) from exc
    return mysql.connector


def q(identifier: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Unsafe identifier: {identifier}")
    return f"`{identifier}`"


def load_data_dictionary(path: Path) -> dict[str, dict[str, str]]:
    if not path.exists():
        return {}
    mapping: dict[str, dict[str, str]] = defaultdict(dict)
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            mapping[row["table_name"]][row["column_name"]] = row["dtype"]
    return mapping


def mysql_type(column: str, dtype: str | None) -> str:
    if column in DATE_COLUMNS or (dtype and "datetime" in dtype):
        return "DATE"
    if dtype and "int" in dtype:
        return "INT"
    if dtype and "float" in dtype:
        return "DOUBLE"
    if column.endswith("_description") or column in {"description", "policy_response"}:
        return "TEXT"
    return "VARCHAR(512)"


def primary_key_for(table: str, columns: list[str]) -> str | None:
    candidates = [column for column in columns if column.endswith("_id") or column.endswith("_key")]
    if candidates and candidates[0] in columns:
        return candidates[0]
    if table.startswith("ops_"):
        candidate = table.removeprefix("ops_").rstrip("s") + "_id"
        return candidate if candidate in columns else None
    return None


def create_table_sql(table: str, columns: list[str], dtypes: dict[str, str]) -> str:
    pk = primary_key_for(table, columns)
    definitions = []
    for column in columns:
        col_type = mysql_type(column, dtypes.get(column))
        nullable = "NOT NULL" if column == pk else "NULL"
        definitions.append(f"  {q(column)} {col_type} {nullable}")
    if pk:
        definitions.append(f"  PRIMARY KEY ({q(pk)})")
    return f"CREATE TABLE IF NOT EXISTS {q(table)} (\n" + ",\n".join(definitions) + "\n) ENGINE=InnoDB;"


def iter_csv_rows(path: Path, columns: list[str]) -> Iterable[tuple[object, ...]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            values: list[object] = []
            for column in columns:
                value = row.get(column, "")
                values.append(None if value == "" else value)
            yield tuple(values)


def load_csv(cursor, table: str, path: Path, dtypes: dict[str, str], replace: bool) -> int:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        columns = list(next(csv.reader(handle)))

    cursor.execute(create_table_sql(table, columns, dtypes))
    if replace:
        cursor.execute(f"TRUNCATE TABLE {q(table)}")

    placeholders = ", ".join(["%s"] * len(columns))
    column_sql = ", ".join(q(column) for column in columns)
    insert_sql = f"INSERT INTO {q(table)} ({column_sql}) VALUES ({placeholders})"

    batch: list[tuple[object, ...]] = []
    total = 0
    for values in iter_csv_rows(path, columns):
        batch.append(values)
        if len(batch) >= 1000:
            cursor.executemany(insert_sql, batch)
            total += len(batch)
            batch.clear()
    if batch:
        cursor.executemany(insert_sql, batch)
        total += len(batch)
    return total


def apply_sql_file(cursor, path: Path, database: str) -> None:
    if not path.exists():
        return
    sql = path.read_text(encoding="utf-8").replace("oil_news_project", database)
    for statement in [part.strip() for part in sql.split(";") if part.strip()]:
        cursor.execute(statement)


def main() -> None:
    parser = argparse.ArgumentParser(description="Load workspace CSV datasets into MySQL.")
    parser.add_argument("--dataset-dir", type=Path, default=DATASET_DIR)
    parser.add_argument("--replace", action="store_true", help="Truncate tables before loading.")
    parser.add_argument("--only", nargs="*", help="Optional list of CSV stem/table names to load.")
    args = parser.parse_args([])

    mysql = require_connector()
    config = get_mysql_config()
    connection = mysql.connect(
        host=config.host,
        port=config.port,
        user=config.user,
        password=config.password,
        autocommit=False,
    )
    cursor = connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {q(config.database)}")
    cursor.execute(f"USE {q(config.database)}")

    dictionary = load_data_dictionary(args.dataset_dir / "data_dictionary.csv")
    csv_files = sorted(args.dataset_dir.glob("*.csv"))
    if args.only:
        wanted = set(args.only)
        csv_files = [path for path in csv_files if path.stem in wanted]

    for csv_path in csv_files:
        table = csv_path.stem
        loaded = load_csv(cursor, table, csv_path, dictionary.get(table, {}), args.replace)
        connection.commit()
        print(f"Loaded {loaded:>6} rows into {table}")

    apply_sql_file(cursor, Path("sql/analytics_views.sql"), config.database)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"Done. Database `{config.database}` is ready.")


if __name__ == "__main__":
    main()



In [ ]:
# Run the load_mysql main function
main()


## 3. Train Oil Price Model
Train the predictive model for Brent crude oil prices.


In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from datetime import date
from pathlib import Path

import joblib
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


FEATURE_COLUMNS = [
    "brent_price_usd",
    "wti_price_usd",
    "dxy_index",
    "vix_index",
    "gpr_index",
    "brent_return",
    "wti_return",
    "brent_lag_1",
    "brent_lag_3",
    "brent_lag_7",
    "wti_lag_1",
    "wti_lag_3",
    "wti_lag_7",
    "brent_volatility_7d",
    "brent_volatility_30d",
    "wti_volatility_7d",
    "wti_volatility_30d",
    "brent_wti_spread",
    "event_severity",
    "event_flag",
]


def to_float(value: str | None) -> float | None:
    if value is None or value == "":
        return None
    try:
        result = float(value)
    except ValueError:
        return None
    return result


def read_market_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    return sorted(rows, key=lambda row: row["market_date"])


def build_examples(
    rows: list[dict[str, str]], horizon: int = 10
) -> tuple[list[list[float]], list[float], list[str]]:
    """Build feature/label pairs where the target is `horizon` trading days ahead."""
    x_rows: list[list[float]] = []
    y_rows: list[float] = []
    dates: list[str] = []
    for idx, row in enumerate(rows[: len(rows) - horizon]):
        future_brent = to_float(rows[idx + horizon].get("brent_price_usd"))
        features = [to_float(row.get(column)) for column in FEATURE_COLUMNS]
        if future_brent is None or any(value is None for value in features):
            continue
        x_rows.append([float(value) for value in features])
        y_rows.append(future_brent)
        dates.append(rows[idx + horizon]["market_date"])
    return x_rows, y_rows, dates


def split_chronological(
    x_rows: list[list[float]], y_rows: list[float], dates: list[str], test_ratio: float
) -> tuple[list[list[float]], list[float], list[str], list[list[float]], list[float], list[str]]:
    split_index = max(1, int(len(x_rows) * (1.0 - test_ratio)))
    return (
        x_rows[:split_index],
        y_rows[:split_index],
        dates[:split_index],
        x_rows[split_index:],
        y_rows[split_index:],
        dates[split_index:],
    )


def metrics(actual: list[float], predicted: list[float]) -> dict[str, float]:
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": mean_squared_error(actual, predicted) ** 0.5,
        "mape_pct": mean_absolute_percentage_error(actual, predicted) * 100,
        "r2": r2_score(actual, predicted),
    }


def write_predictions(
    path: Path,
    dates: list[str],
    actual: list[float],
    predicted: list[float],
    baseline: list[float],
    horizon: int = 10,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["market_date", "actual_brent_future", "predicted_brent_future", "baseline_previous_brent", "horizon_days"])
        for d, a, p, b in zip(dates, actual, predicted, baseline):
            writer.writerow([d, a, p, b, horizon])



def main() -> None:
    parser = argparse.ArgumentParser(description="Train a Brent price model for a configurable horizon ahead.")
    parser.add_argument("--market-csv", type=Path, default=Path("datasets") / "ops_market_daily.csv")
    parser.add_argument("--output-dir", type=Path, default=Path("model_artifacts"))
    parser.add_argument("--test-ratio", type=float, default=0.2)
    parser.add_argument("--alpha", type=float, default=0.1)
    parser.add_argument(
        "--horizon",
        type=int,
        default=10,
        help="Number of trading days ahead to predict (default: 10 ≈ 2 calendar weeks).",
    )
    args = parser.parse_args([])

    rows = read_market_rows(args.market_csv)
    x_rows, y_rows, dates = build_examples(rows, horizon=args.horizon)
    if len(x_rows) < 100:
        raise SystemExit("Not enough model-ready rows to train.")

    train_x, train_y, train_dates, test_x, test_y, test_dates = split_chronological(
        x_rows, y_rows, dates, args.test_ratio
    )
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=args.alpha)),
        ]
    )
    model.fit(train_x, train_y)

    test_predictions = model.predict(test_x).tolist()
    train_predictions = model.predict(train_x).tolist()
    baseline = [row[0] for row in test_x]

    train_metrics = metrics(train_y, train_predictions)
    test_metrics = metrics(test_y, test_predictions)
    baseline_metrics = metrics(test_y, baseline)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    model_path = args.output_dir / "oil_price_model.joblib"
    joblib.dump(model, model_path)
    artifact = {
        "model_type": "sklearn.pipeline.Pipeline(StandardScaler, Ridge)",
        "target": f"{args.horizon}_trading_day_ahead_brent_price_usd",
        "horizon_trading_days": args.horizon,
        "trained_at": date.today().isoformat(),
        "source_file": str(args.market_csv),
        "feature_columns": FEATURE_COLUMNS,
        "model_file": str(model_path),
        "alpha": args.alpha,
        "train_rows": len(train_x),
        "test_rows": len(test_x),
        "train_date_range": [train_dates[0], train_dates[-1]],
        "test_date_range": [test_dates[0], test_dates[-1]],
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "baseline_previous_price_metrics": baseline_metrics,
    }
    (args.output_dir / "oil_price_model.json").write_text(json.dumps(artifact, indent=2), encoding="utf-8")
    write_predictions(args.output_dir / "test_predictions.csv", test_dates, test_y, test_predictions, baseline, horizon=args.horizon)

    print(f"scikit-learn model trained: {args.horizon}-trading-day-ahead Brent price (~2 calendar weeks)")
    print(f"Rows: train={len(train_x)} test={len(test_x)}")
    print(f"Test RMSE: {test_metrics['rmse']:.3f} USD")
    print(f"Test MAE:  {test_metrics['mae']:.3f} USD")
    print(f"Baseline RMSE: {baseline_metrics['rmse']:.3f} USD")
    print(f"Saved: {model_path}")
    print(f"Saved: {args.output_dir / 'oil_price_model.json'}")
    print(f"Saved: {args.output_dir / 'test_predictions.csv'}")


if __name__ == "__main__":
    main()



In [ ]:
# Run the training process
main()


## 4. Forward Price Forecast
Generate a multi-day forward forecast of Brent crude oil prices.


In [ ]:
from __future__ import annotations

import argparse
import csv
import json
from datetime import datetime, timedelta
from pathlib import Path

import joblib


def to_float(value: str | None) -> float:
    if value is None or value == "":
        raise ValueError("Missing numeric input.")
    return float(value)


def estimate_future_trading_date(base_date_str: str, trading_days_ahead: int) -> str:
    """Estimate a calendar date by adding N trading days (skips weekends)."""
    dt = datetime.strptime(base_date_str, "%Y-%m-%d")
    added = 0
    while added < trading_days_ahead:
        dt += timedelta(days=1)
        if dt.weekday() < 5:  # Mon-Fri
            added += 1
    return dt.strftime("%Y-%m-%d")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a forward forecast of Brent crude oil prices."
    )
    parser.add_argument("--metadata", type=Path, default=Path("model_artifacts") / "oil_price_model.json")
    parser.add_argument("--model", type=Path, default=None)
    parser.add_argument("--market-csv", type=Path, default=Path("datasets") / "ops_market_daily.csv")
    args = parser.parse_args([])

    # --- Load model and config ---
    artifact = json.loads(args.metadata.read_text(encoding="utf-8"))
    model_path = args.model or Path(artifact["model_file"])
    horizon: int = artifact.get("horizon_trading_days", 5)
    model = joblib.load(model_path)
    feature_columns = artifact["feature_columns"]

    with args.market_csv.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    sorted_rows = sorted(rows, key=lambda r: r["market_date"])
    last_date = sorted_rows[-1]["market_date"]

    # --- Single-point prediction from the latest row ---
    latest = sorted_rows[-1]
    raw_features = [to_float(latest[col]) for col in feature_columns]
    single_pred = model.predict([raw_features])[0]

    print("=" * 60)
    print("  BRENT CRUDE OIL - FORWARD PRICE FORECAST")
    print("=" * 60)
    print(f"  Last data date  : {last_date}")
    print(f"  Current Brent   : ${float(latest['brent_price_usd']):.2f}")
    print(f"  Model horizon   : {horizon} trading days")
    print(f"  Headline pred.  : ${single_pred:.2f}  ({horizon} days out)")
    print("=" * 60)

    # --- Multi-day forward forecast ---
    # The model predicts price at (source_date + horizon).
    # By feeding the last `horizon` rows of actual data, each row produces
    # a prediction for a different future date:
    #   row at D-horizon+1  →  predicts D+1
    #   row at D-horizon+2  →  predicts D+2
    #   ...
    #   row at D            →  predicts D+horizon
    # This gives us `horizon` predicted future prices.
    forecast_source_rows = sorted_rows[-horizon:]
    forecasts = []

    print(f"\n{'Date':<14} {'Days Ahead':<14} {'Predicted ($)':<16} {'Based On'}")
    print("-" * 60)

    for i, row in enumerate(forecast_source_rows):
        try:
            features = [to_float(row[col]) for col in feature_columns]
        except (ValueError, KeyError):
            continue
        pred = model.predict([features])[0]
        days_ahead = i + 1
        future_date = estimate_future_trading_date(last_date, days_ahead)
        source_date = row["market_date"]
        forecasts.append({
            "forecast_date": future_date,
            "predicted_brent_usd": round(pred, 4),
            "trading_days_ahead": days_ahead,
            "source_date": source_date,
        })
        print(f"{future_date:<14} +{days_ahead:<13} ${pred:<15.2f} {source_date}")

    # --- Save forecast CSV ---
    output_dir = Path(artifact["model_file"]).parent
    forecast_path = output_dir / "forward_forecast.csv"
    output_dir.mkdir(parents=True, exist_ok=True)
    with forecast_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["forecast_date", "predicted_brent_usd", "trading_days_ahead", "source_date"],
        )
        writer.writeheader()
        writer.writerows(forecasts)

    print(f"\nSaved {len(forecasts)}-day forward forecast to {forecast_path}")


if __name__ == "__main__":
    main()



In [ ]:
# Run the forecast
main()


## 5. Visualizations
Generate visualizations of model predictions vs actual prices, including the forward forecast.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates

def create_visualizations(predictions_csv_path, output_dir, forecast_csv_path=None, market_csv_path=None):
    """
    Reads the test predictions and generates graphs comparing actual vs predicted prices.
    Optionally also plots a forward forecast extending beyond actual data.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Loading data from {predictions_csv_path}...")
    try:
        df = pd.read_csv(predictions_csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find {predictions_csv_path}. Please ensure the model has been trained and output data exists.")
        return

    # Convert market_date to datetime
    df['market_date'] = pd.to_datetime(df['market_date'])
    
    # Sort by date just in case
    df = df.sort_values('market_date')

    # Read horizon from CSV if present, otherwise default to 5 trading days
    horizon_days = int(df['horizon_days'].iloc[0]) if 'horizon_days' in df.columns else 5
    horizon_label = f"{horizon_days} Trading Days (~1 Week) Ahead"

    # --- Detect column names (support both old and new schemas) ---
    actual_col = 'actual_brent_future' if 'actual_brent_future' in df.columns else 'actual_brent_next'
    predicted_col = 'predicted_brent_future' if 'predicted_brent_future' in df.columns else 'predicted_brent_next'

    # 1. Line plot of Actual vs Predicted over time
    plt.figure(figsize=(14, 7))
    plt.plot(df['market_date'], df[actual_col], label='Actual Brent Price', color='blue', alpha=0.7, linewidth=1.5)
    plt.plot(df['market_date'], df[predicted_col], label=f'Predicted Brent Price ({horizon_label})', color='orange', alpha=0.8, linewidth=1.5)
    
    plt.title(f'Brent Crude Oil Price: Actual vs Predicted {horizon_label} (Test Set)', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price (USD)', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    # Format x-axis dates
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.gcf().autofmt_xdate()
    
    line_plot_path = os.path.join(output_dir, 'actual_vs_predicted_line.png')
    plt.savefig(line_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved line plot to {line_plot_path}")
    
    # 2. Scatter plot of Actual vs Predicted (to show correlation)
    plt.figure(figsize=(8, 8))
    plt.scatter(df[actual_col], df[predicted_col], alpha=0.5, color='green')
    
    # Add perfect prediction line (y=x)
    min_val = min(df[actual_col].min(), df[predicted_col].min())
    max_val = max(df[actual_col].max(), df[predicted_col].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction (y=x)')
    
    plt.title(f'Prediction Accuracy: Actual vs Predicted ({horizon_label})', fontsize=14)
    plt.xlabel('Actual Price (USD)', fontsize=12)
    plt.ylabel(f'Predicted Price (USD)', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    scatter_plot_path = os.path.join(output_dir, 'actual_vs_predicted_scatter.png')
    plt.savefig(scatter_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved scatter plot to {scatter_plot_path}")
    
    # 3. Line plot including Baseline
    plt.figure(figsize=(14, 7))
    plt.plot(df['market_date'], df[actual_col], label='Actual Brent Price', color='blue', alpha=0.7, linewidth=1.5)
    plt.plot(df['market_date'], df[predicted_col], label=f'Predicted Brent ({horizon_label})', color='orange', alpha=0.8, linewidth=1.5)
    plt.plot(df['market_date'], df['baseline_previous_brent'], label='Baseline (Current Day Price)', color='gray', alpha=0.5, linestyle='--')
    
    plt.title(f'Model vs Baseline vs Actual Prices ({horizon_label})', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price (USD)', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.gcf().autofmt_xdate()
    
    baseline_plot_path = os.path.join(output_dir, 'model_vs_baseline.png')
    plt.savefig(baseline_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved baseline comparison plot to {baseline_plot_path}")

    # 4. Forward Forecast — extends predicted prices beyond actual data
    _plot_forward_forecast(forecast_csv_path, market_csv_path, output_dir, horizon_label)


def _plot_forward_forecast(forecast_csv_path, market_csv_path, output_dir, horizon_label):
    """Plot recent actual prices + forecasted future prices on one chart."""
    if forecast_csv_path is None or market_csv_path is None:
        print("Skipping forward forecast plot (no forecast CSV or market CSV provided).")
        return

    try:
        fc = pd.read_csv(forecast_csv_path)
        market = pd.read_csv(market_csv_path)
    except FileNotFoundError as e:
        print(f"Skipping forward forecast plot: {e}")
        return

    fc['forecast_date'] = pd.to_datetime(fc['forecast_date'])
    market['market_date'] = pd.to_datetime(market['market_date'])
    market = market.sort_values('market_date')

    # Show the last ~60 trading days of actual data for context
    recent = market.tail(60).copy()

    plt.figure(figsize=(14, 7))

    # Plot actual prices
    plt.plot(
        recent['market_date'], recent['brent_price_usd'].astype(float),
        label='Actual Brent Price', color='blue', linewidth=2, alpha=0.8,
    )

    # Bridge point: connect actual line to forecast line
    last_actual_date = recent['market_date'].iloc[-1]
    last_actual_price = float(recent['brent_price_usd'].iloc[-1])

    # Build forecast series with bridge
    fc_sorted = fc.sort_values('forecast_date')
    forecast_dates = pd.concat([pd.Series([last_actual_date]), fc_sorted['forecast_date']], ignore_index=True)
    forecast_prices = pd.concat(
        [pd.Series([last_actual_price]), fc_sorted['predicted_brent_usd'].astype(float)],
        ignore_index=True,
    )

    plt.plot(
        forecast_dates, forecast_prices,
        label='Forecasted Brent Price', color='red', linewidth=2.5,
        linestyle='--', marker='o', markersize=5, alpha=0.9,
    )

    # Shade the forecast region
    plt.axvspan(
        last_actual_date, fc_sorted['forecast_date'].max(),
        alpha=0.08, color='red', label='Forecast Window',
    )

    # Annotate forecast values
    for _, row in fc_sorted.iterrows():
        plt.annotate(
            f"${row['predicted_brent_usd']:.2f}",
            xy=(row['forecast_date'], row['predicted_brent_usd']),
            textcoords="offset points", xytext=(0, 12),
            ha='center', fontsize=8, color='red', fontweight='bold',
        )

    plt.title(f'Brent Crude Oil - Forward Price Forecast ({horizon_label})', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price (USD)', fontsize=12)
    plt.legend(fontsize=11, loc='upper left')
    plt.grid(True, alpha=0.3)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gcf().autofmt_xdate()

    forecast_plot_path = os.path.join(output_dir, 'forward_forecast.png')
    plt.savefig(forecast_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved forward forecast plot to {forecast_plot_path}")



predictions_file = os.path.join('model_artifacts', 'test_predictions.csv')
forecast_file = os.path.join('model_artifacts', 'forward_forecast.csv')
market_file = os.path.join('datasets', 'ops_market_daily.csv')
# Make visualizations display inline instead of saving, or just let them save
%matplotlib inline
create_visualizations(predictions_file, 'Visualization', forecast_file, market_file)

